## Paso 0 — Cargamos las 18 tablas ya tipadas

Mismo patrón que los notebooks anteriores: leemos directo los Parquet
del notebook 00.


In [12]:
import pandas as pd
from pathlib import Path

RUTA_INTERIM = Path("../datos/interim")
ARCHIVOS_PARQUET = sorted(RUTA_INTERIM.glob("*.parquet"))
tablas = {archivo.stem: pd.read_parquet(archivo) for archivo in ARCHIVOS_PARQUET}
assert len(tablas) == 18, f"Se esperaban 18 tablas, se cargaron {len(tablas)}"
print(f"Tablas cargadas: {len(tablas)}")


Tablas cargadas: 18


## Paso 1 — ¿Cuántas líneas Tarifario tienen suficiente info para comparar contra SAP? (§7.2 del plan)

Para comparar el precio cotizado contra el precio de mercado SAP,
necesitamos saber qué LineaNegocio/ClaseViaje usó esa línea — dato que
viene de lineas_solicitud, no de cotizacion_lineas directamente. Ya
sabíamos que esto casi nunca se llena (notebook 01) — aquí confirmamos
cuántas de las 32 líneas Tarifario específicamente tienen ese dato.


In [13]:
cotizacion_lineas = tablas["cotizacion_lineas"]
lineas_solicitud = tablas["lineas_solicitud"]

tarifario = cotizacion_lineas[cotizacion_lineas["fuente"] == "Tarifario"].copy()
tarifario = tarifario.merge(
    lineas_solicitud[["id", "lineaNegocio", "claseViaje", "rutaId"]],
    left_on="lineaSolicitudId", right_on="id", suffixes=("", "_linea")
)

print(f"Líneas con Fuente=Tarifario: {len(cotizacion_lineas[cotizacion_lineas['fuente']=='Tarifario'])}")
con_ambos = tarifario.dropna(subset=["lineaNegocio", "claseViaje"])
print(f"De esas, con LineaNegocio Y ClaseViaje pobladas: {len(con_ambos)}")
print()
print(con_ambos[["id", "lineaNegocio", "claseViaje", "rutaId", "valorFlete"]].to_string())


Líneas con Fuente=Tarifario: 32
De esas, con LineaNegocio Y ClaseViaje pobladas: 0

Empty DataFrame
Columns: [id, lineaNegocio, claseViaje, rutaId, valorFlete]
Index: []


## Paso 2 — Dispersión de VALOR en SAP, por LineaNegocio (§7.3 del plan)

Sin depender del cruce con lo propio: ¿qué tan distinto es el precio de
una tarifa SAP según la categoría de negocio? Esto caracteriza el
mercado por sí mismo, información con valor propio para el futuro
modelo, independiente de si se puede cruzar contra CabinaFletes o no.


In [14]:
tarifa_plaza = tablas["tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy"]

por_linea_negocio = tarifa_plaza.groupby("lineaNegocioDescripcion")["valor"].describe()
print(por_linea_negocio.to_string())



                                              count          mean           std        min        25%        50%        75%         max
lineaNegocioDescripcion                                                                                                                
Oferta de Expedición Devolución Contenedor   1028.0  1.863085e+06  1.439189e+06        0.0   900000.0  1300001.0  2500000.0   8694464.0
Oferta de Expedición Interior a Puerto       2340.0  4.041846e+06  2.270570e+06   220000.0  2232500.0  3600000.0  5500000.0  13272000.0
Oferta de Expedición Licitaciones               1.0  3.556418e+06           NaN  3556418.0  3556418.0  3556418.0  3556418.0   3556418.0
Oferta de Expedición Nacional               13625.0  3.163931e+06  2.038501e+06   100000.0  1600000.0  2700000.0  4200000.0  18000000.0
Oferta de Expedición Puerto a interior       6023.0  4.030362e+06  2.411012e+06   100000.0  2000000.0  3600000.0  5600000.0  15700000.0


## Paso 3 — Dispersión de VALOR en SAP, por tipo de vehículo (§7.3 del plan)

Mismo ejercicio que LineaNegocio, ahora por TIPOVEHICULO (7 categorías,
ya confirmadas 1 a 1 con su descripción en el notebook 04). Misma
salvedad: mezcla variación real de vehículo con variación de ruta.


In [15]:
por_tipo_vehiculo = tarifa_plaza.groupby("vehiculoDescripcion")["valor"].describe()
print(por_tipo_vehiculo.to_string())


                                           count          mean           std       min        25%        50%        75%         max
vehiculoDescripcion                                                                                                                
Dobletroque                                438.0  3.641868e+06  1.390625e+06  700000.0  2800000.0  3500000.0  4600000.0   8000000.0
Liviano (capacidad 4 ton)                 5221.0  1.826780e+06  9.436562e+05  220000.0  1100000.0  1600000.0  2500000.0   7500000.0
Liviano mas 10.5 ton (capacidad 8.5 ton)  5147.0  2.377126e+06  1.191969e+06  100000.0  1400000.0  2200000.0  3224000.0   8500000.0
Mini Mula Poderosa                         421.0  4.686166e+06  1.780740e+06  700000.0  3500000.0  4760000.0  5800000.0   8800000.0
Minimula                                  3488.0  4.087465e+06  1.886331e+06  570000.0  2700000.0  4000000.0  5200000.0  12500000.0
Rígidos, Turbo o Sencillo                   14.0  2.315387e+06  1.371188e+06

## Paso 4 — Dispersión de VALORSICECONHORA en SICE-TAC, por tipo de carga (§7.3 del plan)

Solo sobre `piso-sicetac_nucleo` (126 filas) — las 25 `raras` se dejan
fuera a propósito, son casos extremos elegidos adrede, mezclarlas
sesgaría el resultado hacia arriba/abajo artificialmente.


In [16]:
piso_sicetac_nucleo = tablas["piso-sicetac_nucleo"]
por_tipo_carga = piso_sicetac_nucleo.groupby("nombretipocarga")["valorsiceconhora"].describe()
print(por_tipo_carga.to_string())


                    count          mean           std        min         25%        50%         75%        max
nombretipocarga                                                                                               
Carga Refrigerada     6.0  2.648267e+06  1.502266e+06   913366.0  1434634.25  2619199.5  3617280.25  4748086.0
Contenedor cargado    6.0  4.113858e+06  2.719678e+06  1474516.0  2141309.75  3294614.5  5678825.25  8388967.0
Contenedor vacío      6.0  2.042604e+06  1.682862e+06   330531.0   930772.50  1921094.0  2300937.25  5054689.0
General              24.0  2.568956e+06  1.383970e+06   705009.0  1484076.75  2224079.5  3426629.00  5406168.0
Granel Sólido        30.0  3.050617e+06  1.646392e+06   367451.0  1743835.00  3197555.0  4459708.25  6251385.0
Granel líquido        6.0  4.115099e+06  3.157665e+06  1017941.0  1952659.25  3267965.0  5408030.75  9501111.0
Sin carga            48.0  1.668203e+06  1.366734e+06    39366.0   549494.25  1258556.5  2307198.00  6060042.0


## Paso 5 — Fuente × Resultado de la cotización (§7.4 del plan)

Modalidad × líneas de negocio SAP no se puede — Modalidad tiene solo 1
valor real en las 13 solicitudes. Este otro cruce sí es posible:
¿la fuente que usó CabinaFletes para cotizar tiene alguna relación con
si la cotización se ganó o no? Con 18 cotizaciones, el plan mismo avisa
que cualquier patrón aquí es anecdótico, no estadístico — se reporta
igual, con esa salvedad explícita.


In [17]:
cotizaciones = tablas["cotizaciones"]
cl_con_resultado = cotizacion_lineas.merge(
    cotizaciones[["id", "resultado"]], left_on="cotizacionId", right_on="id", suffixes=("", "_cot")
)
tabla_cruzada = pd.crosstab(cl_con_resultado["fuente"], cl_con_resultado["resultado"])
print(tabla_cruzada)


resultado  Ganada  Perdida  SinDefinir
fuente                                
Manual          2        0           1
Sicetac         2        2           9
Tarifario       8        8          16


## Síntesis del notebook 05 — Bivariado y multivariado (Fase 6, §7 del plan)

**Objetivo del plan:** responder qué tan cerca cotiza CabinaFletes del
piso/plaza de mercado — el corazón de esta HU. Resultado: **no se pudo
responder directamente con este extracto**, pero se caracterizó bien el
mercado por su cuenta, y se dejó clara la razón de por qué no se pudo.

**Cruce propio ↔ mercado (§7.1/§7.2, Paso 1):** 0 casos utilizables.
Solo 5 líneas de solicitud tienen `LineaNegocio`/`ClaseViaje` poblados;
de esas, solo 3 llegaron a cotizarse — y las 3 resultan ser casos ya
conocidos como ruido (la ruta malformada `"b"/"j"`, y los 2 valores
`Fuente=Manual` ya descartados por absurdos). Además, el usuario hizo
notar una falla de método más profunda: comparar solo por código de
negocio (sin ruta) no habría sido válido de todas formas — la misma
combinación de `LineaNegocio`/`ClaseViaje` cubre miles de rutas SAP con
precios muy distintos entre sí. El cruce fino de verdad necesita ruta +
código de negocio juntos, trabajo que le corresponde a la Fase 8.

**Mercado por su cuenta, sin depender de lo propio (§7.3, Pasos 2-4):**
- SAP por `LineaNegocio`: `Nacional` domina (59% del catálogo) y es la
  más barata en promedio; `Puerto a Interior`/`Interior a Puerto` casi
  idénticas entre sí (razonable, son la misma ruta en direcciones
  opuestas); `Licitaciones` tiene solo 1 fila en todo el catálogo.
- SAP por tipo de vehículo: patrón sensato — el precio sube con la
  capacidad (`Liviano 4 ton` la más barata, `Tractomula` la más cara).
  `Rígidos, Turbo o Sencillo` con solo 14 filas, muestra chica.
- SICE-TAC núcleo por tipo de carga: `Sin carga` la más barata,
  `Contenedor cargado`/`Granel líquido` las más caras — también sensato.
  Varias categorías con solo 6 filas cada una (recordar el diseño del
  núcleo: 6 filas por combinación).
- Salvedad transversal: estas dispersiones mezclan variación de negocio
  con variación de ruta — no se pueden separar sin el trabajo de la
  Fase 8.

**Categórica × categórica (§7.4, Paso 5):** `Modalidad` × líneas de
negocio SAP no aplica (Modalidad tiene solo 1 valor real). `Fuente` ×
`Resultado`: calculado (Tarifario y Sicetac ambos 50/50 Ganada/Perdida),
pero **declarado sin señal de negocio** — el usuario confirmó que
`Resultado` también lo marcó él mismo durante pruebas, no refleja una
decisión real de cliente. Se documenta como "no aplicable para inferir
negocio", no como patrón débil.

**Multivariado (§7.5):** no aplicable, tal como anticipaba el plan —
forzar clustering/PCA sobre unas pocas decenas de casos sería
sobreajustar a ruido.

**Qué le queda listo al siguiente notebook:** caracterización sólida del
mercado (SAP/SICE-TAC) por su cuenta, y la conclusión bien fundamentada
de que el cruce fino propio↔mercado no es posible con este piloto — insumo
directo para `hallazgos.md`/`limitaciones.md`. Siguen las Fases 7-11 del
plan (temporal, geográfico, outliers, sesgos, vacíos) antes de la síntesis
final.
